## T10 - Particiones en árboles de clasificación

Sofia Anaya Palafox

Ingeniería Financiera

738594

## Índice GINI

Imagina que estás organizando un grupo de juguetes de colores mezclados. El **índice GINI** es una forma de medir qué tan "mezclado" o "impuro" está tu grupo de juguetes. Si todos tus juguetes son del mismo color, tu grupo es "puro" y el GINI es cero.

En los árboles de clasificación, el GINI se usa para ver qué tan bien una pregunta (una división del árbol) separa los datos en grupos más "puros" (donde la mayoría de los elementos pertenecen a la misma categoría).

**¿Cómo funciona?**

El árbol busca hacer divisiones donde los nuevos grupos tengan el GINI más bajo posible. Así, cada "rama" del árbol se vuelve más específica para una categoría.

**Conexión con las Particiones:**
Cuando un árbol de clasificación decide cómo dividir los datos (crear una "partición"), usa el índice GINI para encontrar la mejor división. La mejor división es aquella que hace que los nuevos grupos resultantes sean lo más "puros" posible, es decir, que tengan el menor GINI.

## Entropía

La **Entropía** es muy parecida al GINI, pero usa un concepto de "desorden" o "incertidumbre". Cuanta más entropía tiene un grupo, más mezcladas están las categorías y más difícil es predecir a qué categoría pertenece un elemento.

Piensa en un mazo de cartas. Si todas las cartas son del mismo palo, hay poca entropía (sabes que la siguiente carta será de ese palo). Si las cartas están muy mezcladas, hay mucha entropía (no sabes qué palo saldrá).

**¿Cómo funciona?**

En los árboles de clasificación, se busca reducir la entropía. Cada división del árbol intenta disminuir la incertidumbre en los grupos resultantes. La cantidad en que se reduce la entropía se llama "ganancia de información", y el árbol busca maximizarla en cada paso.

**Conexión con las Particiones:**
La entropía también guía la creación de particiones en un árbol de clasificación. El árbol busca divisiones que disminuyan la entropía en los nodos hijos, lo que significa que los datos dentro de cada nueva partición están más organizados y son más fáciles de clasificar.

## Log Loss (Pérdida Logarítmica o Pérdida de Entropía Cruzada)

El **Log Loss** es diferente del GINI y la Entropía porque no se usa para construir el árbol (para decidir cómo dividir los datos). En cambio, se usa para ver qué tan bien funciona un modelo de clasificación después de que ya está construido, especialmente cuando el modelo da una probabilidad de que algo pertenezca a una categoría.

**¿Cómo funciona?**

"Castiga" fuertemente las predicciones que son muy seguras pero incorrectas. Un Log Loss bajo significa que el modelo es bueno y sus probabilidades son precisas. Un Log Loss alto significa que el modelo no es bueno.

**Conexión con las Particiones:**
El Log Loss no influye directamente en cómo se hacen las particiones *dentro de un solo árbol de decisión*. Sin embargo, en modelos más complejos que usan muchos árboles (como los "Bosques Aleatorios" o "Gradient Boosting"), el Log Loss puede usarse como una medida para entrenar esos modelos combinados, haciendo que los árboles se ajusten para minimizar este error globalmente. Pero para un solo árbol simple, GINI y Entropía son los que guían las divisiones.

## ¿Cuál es la diferencia entre Entropía y Log Loss?

Aunque ambos usan logaritmos y miden algo parecido a la incertidumbre, su uso es distinto:

1.  **Propósito:**
    *   **Entropía:** Se usa para **construir el árbol de clasificación**. Ayuda a decidir dónde y cómo dividir los datos para que cada nueva rama sea menos mezclada.
    *   **Log Loss:** Se usa para **medir qué tan bueno es el modelo** después de que ya está entrenado. Evalúa las probabilidades que el modelo predice comparándolas con la realidad.

2.  **Qué miden:**
    *   **Entropía:** Mide el **desorden o la mezcla de las categorías**.
    *   **Log Loss:** Mide el **error de las predicciones de probabilidad**

3.  **En (mis palabras)**
    *   **Entropía:** Es una medida para **guiar el crecimiento del árbol** durante el entrenamiento.
    *   **Log Loss:** Es una **métrica de evaluación** para calificar el rendimiento final.

## Ejercicio Práctico

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel('/content/Motor Trend Car Road Tests (2).xlsx')
print("Archivo cargado correctamente.")

print("\nPrimeras 5 filas del dataset:")
display(df.head())
print("\nInformación del DataFrame:")
df.info()

Archivo cargado correctamente.

Primeras 5 filas del dataset:


,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2



Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


In [ ]:
# 'am' como variable (y)
y = df['am']
# 'hp' como variable (X)
X = df['hp']

print(f"Variable Objetivo (y): {y.name} (0=Automático, 1=Manual)")
print(f"Variable Predictora (X): {X.name} (Caballos de fuerza)")

# Contamos las ocurrencias de la variable objetivo en el nodo raíz (antes de cualquier división)
"Este código está analizando el nodo raíz de un árbol de decisión (el nodo"
"inicial, antes de cualquier división) para ver cómo están distribuidas las clases."
class_counts_root = y.value_counts()
total_samples_root = len(y)

print(f"\nConteo de clases en el nodo raíz:\n{class_counts_root}")
print(f"Total de muestras en el nodo raíz: {total_samples_root}")

# Calculamos las proporciones de las clases en el nodo raíz
prob_class_0_root = class_counts_root[0] / total_samples_root
prob_class_1_root = class_counts_root[1] / total_samples_root

print(f"Proporción de clase 0 (Automático): {prob_class_0_root:.2f}")
print(f"Proporción de clase 1 (Manual): {prob_class_1_root:.2f}")

Variable Objetivo (y): am (0=Automático, 1=Manual)
Variable Predictora (X): hp (Caballos de fuerza)

Conteo de clases en el nodo raíz:
am
0    19
1    13
Name: count, dtype: int64
Total de muestras en el nodo raíz: 32
Proporción de clase 0 (Automático): 0.59
Proporción de clase 1 (Manual): 0.41


### 1. GINI

El índice GINI mide la impureza de un nodo. Un GINI de 0 significa que el nodo es perfectamente puro. Un valor más alto indica más mezcla.

**Fórmula GINI:** $GINI = 1 - \sum_{i=1}^{C} (p_i)^2$


In [ ]:
def calculate_gini(y_subset):
    if len(y_subset) == 0:
        return 0
    class_counts = y_subset.value_counts()
    total_samples = len(y_subset)
    gini = 1.0
    for count in class_counts:
        p = count / total_samples
        gini -= p**2
    return gini

# 1. GINI del nodo raíz
gini_root = calculate_gini(y)
print(f"GINI del nodo raíz: {gini_root:.3f}")

# 2. Elegir un punto de división (por ejemplo, horsepower <= 150)
split_point = 150
left_node_mask = X <= split_point
right_node_mask = X > split_point

y_left = y[left_node_mask]
y_right = y[right_node_mask]

print(f"\nDivisión propuesta: hp <= {split_point}")
print(f"Tamaño del nodo izquierdo (hp <= {split_point}): {len(y_left)} muestras")
print(f"Tamaño del nodo derecho (hp > {split_point}): {len(y_right)} muestras")

# 3. GINI de los nodos hijos
gini_left = calculate_gini(y_left)
gini_right = calculate_gini(y_right)

print(f"GINI del nodo izquierdo: {gini_left:.3f}")
print(f"GINI del nodo derecho: {gini_right:.3f}")

# 4. GINI ponderado de la división
weighted_gini_split = (len(y_left) / total_samples_root) * gini_left + \
                      (len(y_right) / total_samples_root) * gini_right

print(f"GINI ponderado de la división: {weighted_gini_split:.3f}")

# 5. Reducción de impureza GINI (Information Gain GINI)
reduction_gini = gini_root - weighted_gini_split
print(f"Reducción de impureza GINI: {reduction_gini:.3f}")

print("\nUn valor de reducción de impureza GINI positivo indica que esta división mejora la pureza de los nodos. El árbol elegiría la división que maximiza esta reducción.")

GINI del nodo raíz: 0.482

División propuesta: hp <= 150
Tamaño del nodo izquierdo (hp <= 150): 19 muestras
Tamaño del nodo derecho (hp > 150): 13 muestras
GINI del nodo izquierdo: 0.499
GINI del nodo derecho: 0.355
GINI ponderado de la división: 0.440
Reducción de impureza GINI: 0.042

Un valor de reducción de impureza GINI positivo indica que esta división mejora la pureza de los nodos. El árbol elegiría la división que maximiza esta reducción.


### 2. Demostración de la Entropía para una Partición

La Entropía mide el nivel de desorden o incertidumbre en un nodo.

**Fórmula Entropía:** $Entropy = -\sum_{i=1}^{C} p_i \log_2(p_i)$



In [ ]:
def calculate_entropy(y_subset):
    if len(y_subset) == 0:
        return 0
    class_counts = y_subset.value_counts()
    total_samples = len(y_subset)
    entropy = 0.0
    for count in class_counts:
        p = count / total_samples
        if p > 0: # Evitar log(0)
            entropy -= p * np.log2(p)
    return entropy

# 1. Entropía del nodo raíz
entropy_root = calculate_entropy(y)
print(f"Entropía del nodo raíz: {entropy_root:.3f}")

# Usamos la misma división que para GINI
# 2. y 3. Entropía de los nodos hijos
entropy_left = calculate_entropy(y_left)
entropy_right = calculate_entropy(y_right)

print(f"Entropía del nodo izquierdo: {entropy_left:.3f}")
print(f"Entropía del nodo derecho: {entropy_right:.3f}")

# 4. Entropía ponderada de la división
weighted_entropy_split = (len(y_left) / total_samples_root) * entropy_left + \
                         (len(y_right) / total_samples_root) * entropy_right

print(f"Entropía ponderada de la división: {weighted_entropy_split:.3f}")

# 5. Ganancia de Información
information_gain = entropy_root - weighted_entropy_split
print(f"Ganancia de Información (Information Gain): {information_gain:.3f}")

print("\nUna ganancia de información positiva significa que esta división ha reducido el desorden en los nodos hijos. ")

Entropía del nodo raíz: 0.974
Entropía del nodo izquierdo: 0.998
Entropía del nodo derecho: 0.779
Entropía ponderada de la división: 0.909
Ganancia de Información (Information Gain): 0.065

Una ganancia de información positiva significa que esta división ha reducido el desorden en los nodos hijos. 


### 3. Demostración de Log Loss (Pérdida Logarítmica)

Como mencionamos, el Log Loss no se usa para *dividir* el árbol, sino para *evaluar* qué tan buenas son las predicciones de probabilidad de un modelo. Para demostrarlo, primero necesitamos:

1.  **Entrenar un modelo de clasificación simple:** Usaremos un `DecisionTreeClassifier` de `sklearn` para predecir si la transmisión (`am`) es automática (0) o manual (1) basándonos en la potencia (`hp`).
2.  **Obtener las probabilidades predichas:** El modelo nos dará la probabilidad de que cada coche sea de transmisión manual.
3.  **Calcular el Log Loss:** Usaremos la función `log_loss` de `sklearn.metrics` para comparar estas probabilidades con las etiquetas reales.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

X_model = df[['hp']]

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_model, y, test_size=0.3, random_state=42)

# 1. Entrenar un modelo de árbol de decisión simple
# Max_depth=3 para un árbol no tan difícil
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("Modelo de Árbol de Decisión entrenado.")

# 2. Obtener las probabilidades predichas para el conjunto de prueba
probabilities = model.predict_proba(X_test)

# Las probabilidades de la clase positiva (manual = 1) están en la segunda columna
predicted_probabilities_class_1 = probabilities[:, 1]

print("\nPrimeras 5 probabilidades predichas (para clase 1 - manual):\n", predicted_probabilities_class_1[:5])
print("Primeras 5 etiquetas reales (para el conjunto de prueba):\n", y_test.head())

# 3. Calcular el Log Loss
ll = log_loss(y_test, predicted_probabilities_class_1)

print(f"\nLog Loss del modelo: {ll:.3f}")

print("\nUn valor de Log Loss cercano a 0 indica que el modelo hace predicciones de probabilidad muy precisas. Cuanto mayor sea el valor, menos precisas son las probabilidades del modelo (o más segura es la predicción pero incorrecta).")

Modelo de Árbol de Decisión entrenado.

Primeras 5 probabilidades predichas (para clase 1 - manual):
 [0. 0. 0. 1. 1.]
Primeras 5 etiquetas reales (para el conjunto de prueba):
 29    1
15    0
24    0
17    1
8     0
Name: am, dtype: int64

Log Loss del modelo: 7.237

Un valor de Log Loss cercano a 0 indica que el modelo hace predicciones de probabilidad muy precisas. Cuanto mayor sea el valor, menos precisas son las probabilidades del modelo (o más segura es la predicción pero incorrecta).


### Resumen Final de las Métricas

-   **GINI / Entropía:** Son criterios internos de un árbol de decisión para decidir cómo hacer las *divisiones* (particiones) en cada nodo. Buscan hacer que los grupos resultantes sean lo más puros u homogéneos posible.
    -   Calculamos una **reducción de impureza GINI** de **0.042** y una **ganancia de información** (basada en Entropía) de **0.065** para nuestro punto de división de ejemplo (`hp <= 150`). Valores positivos indican una mejora.

-   **Log Loss:** Es una métrica de *evaluación externa* que mide la precisión de las *predicciones de probabilidad* de un modelo ya entrenado. Castiga las predicciones seguras pero incorrectas.
    -   Nuestro modelo simple de árbol de decisión obtuvo un **Log Loss de 7.237**. Este valor nos dice qué tan bien se alinean las probabilidades predichas por el modelo con los resultados reales.